In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import math

In [5]:
data = pd.read_csv("../new_code/DATASET.csv")

In [6]:
data.head()

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,Time to Depletion,type,capacity,charged
0,9.36,11.84,0.156000,0.156000,110.8224,26.844000,10324.615385,b2,88.81,27.0
1,9.34,11.84,0.155667,0.311667,110.5856,26.688333,10286.723769,b2,88.81,27.0
2,9.34,11.83,0.155667,0.467333,110.4922,26.532667,10226.723769,b2,88.81,27.0
3,7.14,11.88,0.119000,0.586333,84.8232,26.413667,13317.815126,b2,88.81,27.0
4,7.13,11.88,0.118833,0.705167,84.7044,26.294833,13276.493689,b2,88.81,27.0


In [7]:
data.shape

(5437, 10)

In [8]:
TARGET_VARIABLE = 'Time to Depletion'

In [9]:
Y = data[TARGET_VARIABLE]
X = data.drop(TARGET_VARIABLE, axis=1)

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical Featrures are : ", numerical_features)
print("Categorical Featrures are : ", categorical_features)

Numerical Featrures are :  ['Current', 'Voltage', 'Ah Out', 'Cumulative Actual Disch Ah', 'Power', 'Remaining Capacity', 'capacity', 'charged']
Categorical Featrures are :  ['type']


In [10]:
numerical_transformer = Pipeline(steps=[
    ('pass',
     'passthrough')
])
categorical_transformer = Pipeline(steps=[
    ('onehot',
     OneHotEncoder(handle_unknown='ignore',
                   sparse_output=False))
])

In [11]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
]
    ,remainder='passthrough')

In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

In [13]:
type(X_train)

pandas.core.frame.DataFrame

In [14]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [15]:
X_test

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,type,capacity,charged
3204,8.47,10.62,0.141167,85.515000,89.9514,-0.515000,b2,85.00,85.00
4515,7.50,12.33,0.125000,16.162458,92.4750,68.837542,b5,85.00,85.00
2146,9.51,11.79,0.158500,44.043500,112.1229,44.766500,b2,88.81,88.81
3105,24.11,11.30,0.401833,63.895833,272.4430,21.104167,b2,85.00,85.00
881,17.42,11.77,0.290333,26.487167,205.0334,54.792833,b1,81.28,81.28
...,...,...,...,...,...,...,...,...,...
4333,8.32,11.09,0.138667,27.725167,92.2688,8.274833,b1,81.84,36.00
3254,17.36,12.28,0.289333,1.066167,213.1808,87.283833,b3,88.35,88.35
2357,4.41,9.56,0.073500,85.251833,42.1596,3.558167,b2,88.81,88.81
4990,1.65,10.70,0.027500,81.538292,17.6550,3.461708,b5,85.00,85.00


In [16]:
X_test_processed

array([[ 8.47      , 10.62      ,  0.14116667, ...,  0.        ,
         0.        ,  0.        ],
       [ 7.5       , 12.33      ,  0.125     , ...,  0.        ,
         1.        ,  0.        ],
       [ 9.51      , 11.79      ,  0.1585    , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 4.41      ,  9.56      ,  0.0735    , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.65      , 10.7       ,  0.0275    , ...,  0.        ,
         1.        ,  0.        ],
       [ 1.75      , 10.35      ,  0.02916667, ...,  0.        ,
         1.        ,  0.        ]])

In [17]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
)

In [18]:
param_grid = {
    'n_estimators' : [50,100,150,200],
    'learning_rate': [0.05,0.1,0.2,0.3],
    'max_depth': [3,5,7,10,12],
    'subsample': [0.5,0.6,0.7,0.8,0.9],
    'colsample_bytree': [0.5,0.6,0.7,0.8,0.9]
}

In [19]:
grid_search = GridSearchCV(
    estimator = xgb_model,
    param_grid = param_grid,
    scoring = 'neg_mean_absolute_error',
    cv = 5,
    verbose = 1,
    n_jobs = -1,
    return_train_score = True,
    refit = True
)

In [20]:
print("Initiating the Grid Search...")
grid_search.fit(X_train_processed, Y_train)
print("Search Finished")

Initiating the Grid Search...
Fitting 5 folds for each of 2000 candidates, totalling 10000 fits


KeyboardInterrupt: 

In [ ]:
best_match = grid_search.best_estimator_

In [122]:
Y_pred = best_match.predict(X_test_processed)

In [123]:
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error is : {mae:.2f}")

Mean Absolute Error is : 160.49


In [124]:
best_match.save_model("../models/battery_xgboost_model.json")

In [125]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("../models/battery_xgboost_model.json")

In [126]:
y_loaded_pred = loaded_model.predict(X_test_processed)

mae_loaded = mean_absolute_error(Y_test, y_loaded_pred)
print(f"Mean Absolute Error is : {mae_loaded:.2f}")

Mean Absolute Error is : 160.49
